In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#Loading the API key from .env file
load_dotenv()

#Initializing the models
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
summarizer_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [2]:
#System prompt for travel bot
travel_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a travel assistant. "
     "You only answer travel-related questions such as flights, hotels, destinations, "
     "itineraries, budgets, travel tips, and packing advice. "
     "For non-travel questions, respond with exactly: 'I can't help with it.'"
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{user_input}")
])

In [3]:
#Prompt for summarizer
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a conversation summarizer for a travel assistant.\n\n"
     "Summarize the conversation into a compact memory.\n"
     "Include:\n"
     "- User preferences (budget, dates, airports, hotel style)\n"
     "- Confirmed trip details\n"
     "- Constraints (kids, pets, driving vs flying)\n"
     "- Open questions\n\n"
     "Do NOT invent facts."
    ),
    ("human",
     "Previous summary:\n{previous_summary}\n\n"
     "New conversation content:\n{conversation_text}\n\n"
     "Write an updated merged summary."
    )
])

In [4]:
#Initializing the variables
chat_history = []         # will hold HumanMessage / AIMessage
running_summary = ""
conversation_count = 0

In [5]:
def chat(user_input):         #Defining function for chat conversation
    global chat_history, running_summary, conversation_count

    # Build messages from template
    messages = travel_prompt.format_messages(
        chat_history=chat_history,
        user_input=user_input
    )

    print("Bot: ", end="", flush=True)

    chunks = []
    for chunk in llm.stream(messages):
        text = getattr(chunk, "content", "")
        if text:
            print(text, end="", flush=True)
            chunks.append(text)

    print("\n")

    full_reply = "".join(chunks).strip()

    # Persist conversation
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=full_reply))
    conversation_count += 1

    # Summarize every 5 turns
    if conversation_count % 5 == 0:
        compress_memory()

In [6]:
def compress_memory(keep_last=2):
    global chat_history, running_summary

    if len(chat_history) <= keep_last:
        return

    old_messages = chat_history[:-keep_last]

    conversation_text = "\n".join(
        f"USER: {m.content}" if isinstance(m, HumanMessage) else f"ASSISTANT: {m.content}"
        for m in old_messages
    )

    summary_messages = summary_prompt.format_messages(
        previous_summary=running_summary,
        conversation_text=conversation_text
    )

    running_summary = summarizer_llm.invoke(summary_messages).content.strip()

    # Replace old history with memory
    chat_history = [
        SystemMessage(content=f"Conversation memory:\n{running_summary}")
    ] + chat_history[-keep_last:]

    print("🧠 Memory summarized & compressed.\n")

In [7]:
chat("I want a budget-friendly 3-day trip from Dallas to Destin")

Bot: Here's a budget-friendly 3-day trip plan from Dallas to Destin:

**Budget-Friendly Tips for Your Trip:**

1.  **Transportation:**
    *   **Drive:** This is by far the most budget-friendly option for a 3-day trip. The drive from Dallas to Destin is about 10-11 hours, so plan for a long travel day on both ends. Split gas costs if traveling with others.
    *   **Pack a Cooler:** Bring drinks, snacks, and sandwich supplies for the drive and your beach days to avoid expensive convenience store stops and restaurant lunches.

2.  **Accommodation:**
    *   **Look Off-Beach:** Hotels or motels slightly off the main beachfront strip, or in nearby towns like Fort Walton Beach or Niceville, can be significantly cheaper.
    *   **Vacation Rentals:** If traveling with a group, consider an Airbnb or VRBO with a kitchen to save on food costs.
    *   **Amenities:** Prioritize places with a kitchenette or at least a mini-fridge and microwave.

3.  **Food:**
    *   **Grocery Store:** Make a gr

In [10]:
chat("I prefer driving and staying near the beach")

Bot: Okay, I understand! Driving is a great budget choice, and staying near the beach is definitely the best way to experience Destin.

Here's a revised budget-friendly 3-day trip plan from Dallas to Destin, focusing on finding accommodation as close to the beach as possible without breaking the bank:

**Key Budget-Friendly Strategies (with your preferences in mind):**

1.  **Transportation (Driving):**
    *   The drive from Dallas to Destin is 10-11 hours. Plan for one full day of travel on each end.
    *   **Fuel Efficiency:** Ensure your car is in good shape for the long drive.
    *   **Road Trip Snacks:** Pack a large cooler with drinks, snacks, sandwiches, and fruit for the drive and your beach days. This is a huge money saver compared to gas station stops and beachside vendors.
    *   **Split Costs:** If traveling with others, split gas costs.

2.  **Accommodation (Near the Beach - Budget Focus):**
    *   **Expand Your Search:** While "Destin" is popular, look at **Fort Walt

In [13]:
chat("Suggest some cheap food options")

Bot: Great question! Eating affordably is key to a budget-friendly trip, especially when staying near the beach. Here are some cheap food options for your Destin trip:

**1. Maximize Grocery Store Purchases (Your Best Friend for Savings!):**

*   **Breakfasts:**
    *   Oatmeal packets, cereal and milk, instant coffee/tea.
    *   Yogurt, fruit (bananas, apples, oranges).
    *   Bagels/bread with cream cheese or peanut butter.
    *   Eggs (if your accommodation has a kitchenette).
*   **Lunches:**
    *   **Sandwich Supplies:** Bread, deli meat, cheese, lettuce, tomatoes, mustard/mayo. Make these in advance and pack them in your cooler for the beach.
    *   **Wraps:** Tortillas, canned tuna/chicken, pre-shredded coleslaw mix.
    *   **Snacks:** Granola bars, crackers, fruit, pretzels, nuts.
    *   **Drinks:** Water bottles, juice boxes, soda. Buying a case of water is far cheaper than individual bottles.
*   **Dinners (if you have a microwave/kitchenette):**
    *   Frozen meals (

In [16]:
chat("What is Machine Learning")

Bot: I can't help with it.



In [18]:
chat("What all you can help with?")

Bot: I can help with travel-related questions such as:

*   **Flights:** Finding flights, understanding airline policies, tips for booking.
*   **Hotels & Accommodation:** Suggestions for hotels, motels, vacation rentals, tips for booking.
*   **Destinations:** Information about various travel destinations, things to do, best times to visit.
*   **Itineraries:** Planning trip schedules and activities for specific destinations.
*   **Budgets:** Advice on how to save money while traveling, estimating trip costs.
*   **Travel Tips:** General advice for travelers, safety tips, cultural etiquette.
*   **Packing Advice:** What to pack for different types of trips or destinations.

🧠 Memory summarized & compressed.



In [19]:
print(conversation_count)

5


In [21]:
print(chat_history)

[SystemMessage(content="Conversation memory:\nHere's a summary of the conversation:\n\n**User Preferences:**\n*   **Destination:** Destin, FL\n*   **Origin:** Dallas, TX\n*   **Duration:** 3-day trip\n*   **Budget:** Budget-friendly\n*   **Transportation:** Driving\n*   **Accommodation:** Near the beach (specifically looking at Fort Walton Beach or Okaloosa Island for more budget-friendly options)\n\n**Confirmed Trip Details:**\n*   **Transportation:** Driving from Dallas to Destin (approx. 10-11 hours each way). User confirmed preference for driving.\n*   **Accommodation:** Focus on budget-friendly options near the beach, including older motels/hotels or small condo rentals (especially if traveling with a group) with kitchen facilities (mini-fridge, microwave, or full kitchen).\n*   **Food Strategy:** Emphasize grocery shopping upon arrival, packing a cooler for snacks/lunches, picnics on the beach, and choosing casual/affordable dining spots for dinners.\n*   **Activities:** Primaril

In [20]:
print(len(chat_history))

7
